<a href="https://colab.research.google.com/github/trinhtattran/RAGassistant/blob/main/ParkinsonsDisease_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive #Section 1
drive.mount('/content/drive')  # prompts for authorisation

# install the LangChain ecosystem and other tools
!pip install pypdf
!pip install -U langchain-core langchain-community langchain-text-splitters \
               sentence-transformers chromadb gradio pypdf

In [ ]:
DATA_DIR = "/content/drive/MyDrive/RAG/data"

In [ ]:
from pathlib import Path #section 2
from langchain_community.document_loaders import PyPDFLoader

def load_pdfs(data_dir: str):
    pdf_paths = list(Path(data_dir).glob('*.pdf'))
    documents = []
    for path in pdf_paths:
        loader = PyPDFLoader(str(path))
        pages = loader.load()  # one Document per page
        for page in pages:
            page.metadata['source'] = path.name
        documents.extend(pages)
    return documents

raw_docs = load_pdfs(DATA_DIR)
print(f"Loaded {len(raw_docs)} pages from {len(set(d.metadata['source'] for d in raw_docs))} PDFs")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #section 3

def chunk_documents(documents):
    # Larger chunks with overlap for better context
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    return splitter.split_documents(documents)

chunks = chunk_documents(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} pages")

In [ ]:
from sentence_transformers import SentenceTransformer #section 4
import chromadb

def create_vector_db(chunks, persist_directory="/content/drive/MyDrive/RAG/chroma_db"):
    texts = [doc.page_content for doc in chunks]
    metadatas = [doc.metadata for doc in chunks]
    ids = [f"chunk_{i}" for i in range(len(chunks))]

    # Load the embedding model
    embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

    # Create a persistent Chroma client and collection
    client = chromadb.PersistentClient(path=persist_directory)
    collection = client.get_or_create_collection(name="parkinsons_docs")
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=metadatas
    )
    return client, collection, embed_model

# Build the vector database once
client, collection, embed_model = create_vector_db(chunks)

In [ ]:
def retrieve(query: str, top_k: int = 8, model=embed_model, coll=collection): #section 5
    """
    Embed the query, search the vector store and return the top‑K documents and metadata.
    """
    query_embedding = model.encode([query])
    results = coll.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas"]
    )
    return results['documents'][0], results['metadatas'][0]

# Test retrieval to inspect the context
docs, metas = retrieve("What are the motor symptoms of Parkinson's disease?", top_k=8)
for d, m in zip(docs, metas):
    print(m['source'], m.get('page', 'unknown'), "→", d[:120], "…")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM #section 6

# Load a larger generation model (requires more RAM)
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-large')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-large')

def answer_question(query: str, top_k: int = 8, max_new_tokens: int = 256):
    # Retrieve relevant chunks
    docs, metas = retrieve(query, top_k=top_k)
    # Build the context string with citation markers
    context_lines = []
    citations = []
    for i, (doc_text, meta) in enumerate(zip(docs, metas)):
        label = f"[{i+1}]"
        context_lines.append(
            f"{label} Source ({meta['source']}, page {meta.get('page', 'unknown')}): {doc_text[:800]}"
        )
        citations.append(label + f" {meta['source']}")
    context = "\n\n".join(context_lines)
    # Refined prompt requesting a bullet list
    prompt = (
        "You are a clinical assistant specialised in Parkinson's disease. "
        "Using only the provided context, list all motor symptoms of Parkinson's disease. "
        "Format your answer as a concise bullet list with each symptom on its own line. "
        "If the context does not contain the answer, respond with 'I don't know'.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=2048,  # allow longer input for larger context
        truncation=True
    )
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )
    raw_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    citation_text = "\n\nSources: " + ", ".join(citations)
    return raw_answer.strip() + citation_text

# Example usage
print(answer_question("What are the motor symptoms of Parkinson's disease?"))

In [ ]:
print(answer_question(
    "What is the Unified Parkinson’s Disease Rating Scale (UPDRS)?",
    top_k=12,
    max_new_tokens=256
))

In [ ]:
docs, metas = retrieve("UPDRS parts", top_k=12)
for meta, doc in zip(metas, docs):
    print(meta['source'], doc[:200])

In [ ]:
import gradio as gr

def qa_function(query):
    return answer_question(query, top_k=15)

iface = gr.Interface(
    fn=qa_function,
    inputs=gr.Textbox(lines=2, label="Ask a question about Parkinson's disease"),
    outputs=gr.Textbox(lines=10, label="Answer with citations"),
    title="Parkinson's Disease RAG Assistant"
)
iface.launch(share=True)